[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/02_inter_annotator_agreement/02_inter_annotator_agreement.ipynb)

# 02 · 标注者一致性 IAA（从零实现）

目标：把「两个/多个标注者有多一致」从一个含糊的印象，变成 **Cohen κ / Fleiss κ / Krippendorff α** 这几个精确的数。用 numpy 从零：① 观察一致率及其陷阱；② Cohen κ（两固定标注者）；③ Fleiss κ（多标注者）；④ Krippendorff α（重合矩阵）；⑤ **prevalence 悖论**（高一致率却低 κ）；⑥ α 的**有序 vs 名义**度量。

路线：观察一致率 → Cohen κ → Fleiss κ → Krippendorff α → κ 悖论 → 有序度量 → ✏️ 练习 → 📖 答案 → 🧪 真实仇恨言论多标注者胶囊。

> 纪律：每个系数都**用两种等价算法对拍**（如 κ 既从列联表又从标注序列算），或在**已知答案的手算例子/标准基准**上验证；完美一致 → κ=α=1；独立瞎标 → κ≈α≈0。所有「应成立的性质」写成 `assert`。

## 0 · 数据 helper（联网取真实数据，失败回退）

下面这个 cell 定义全课统一的下载工具，真实数据胶囊会用到。

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · 观察一致率 $p_o$ 及其陷阱

最朴素的一致性：两个标注者标得**一样**的比例 `p_o = (y1==y2).mean()`。

它的**致命缺陷**：不扣「碰巧一致」。下面构造一个**高度不平衡**的任务，两个**各自瞎猜**（不看样本）的标注者也能撞出很高的 `p_o`——这说明高 `p_o` 可能毫无真实共识。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def observed_agreement(y1, y2):
    '''观察一致率：两标注者标得相同的比例。'''
    y1, y2 = np.asarray(y1), np.asarray(y2)
    return (y1 == y2).mean()

# 不平衡任务: 95% 的样本是类0(非仇恨)。两个标注者都'瞎猜'，但都按 95/5 的偏好猜
N = 5000
r1 = (rng.random(N) > 0.95).astype(int)   # 各自独立瞎标，偏向类0
r2 = (rng.random(N) > 0.95).astype(int)
po_chance = observed_agreement(r1, r2)
print(f'两个「瞎猜」标注者(无真实共识)的观察一致率 p_o = {po_chance:.3f}')
print('-> 高达 0.9 左右，但这全是巧合！因为一类太常见，瞎猜也常撞上')
assert po_chance > 0.85, '不平衡下瞎猜也有高 p_o'

# 对比: 完美一致
yp = rng.integers(0, 2, N)
assert observed_agreement(yp, yp) == 1.0
print('✅ 观察一致率会被「碰巧一致」严重高估 —— 这就是为什么需要偶然校正')

## 2 · Cohen κ：扣掉碰巧一致（两个固定标注者）

$$\kappa = \frac{p_o - p_e}{1 - p_e},\qquad p_e = \sum_i \hat p^A_i\,\hat p^B_i$$

$p_e$ = 两标注者各按**自己的边缘分布**独立瞎标的偶然一致率。我们**用两种等价算法**实现并对拍：一种从两条标注序列直接算，一种从 $K\times K$ 列联表算。

In [ ]:
def cohen_kappa_from_labels(y1, y2, K=None):
    '''从两条标注序列算 Cohen κ。'''
    y1, y2 = np.asarray(y1), np.asarray(y2)
    if K is None: K = int(max(y1.max(), y2.max())) + 1
    N = len(y1)
    po = (y1 == y2).mean()
    pa = np.bincount(y1, minlength=K) / N      # A 的边缘分布
    pb = np.bincount(y2, minlength=K) / N      # B 的边缘分布
    pe = np.sum(pa * pb)
    return (po - pe) / (1 - pe)

def contingency_table(y1, y2, K):
    '''C[i,j] = #(A 标 i 且 B 标 j)。'''
    C = np.zeros((K, K))
    for a, b in zip(y1, y2):
        C[a, b] += 1
    return C

def cohen_kappa_from_table(C):
    '''从列联表算 Cohen κ（另一条等价路径）。'''
    C = np.asarray(C, float); N = C.sum()
    po = np.trace(C) / N
    row = C.sum(axis=1) / N                    # A 边缘
    col = C.sum(axis=0) / N                    # B 边缘
    pe = np.sum(row * col)
    return (po - pe) / (1 - pe)

# 讲解里的手算例子: 都标非70, 都标仇恨10, A非B仇恨5, A仇恨B非15
C = np.array([[70, 5], [15, 10]], float)      # 行=A(非,仇恨), 列=B(非,仇恨)
y1, y2 = [], []
for i in range(2):
    for j in range(2):
        y1 += [i]*int(C[i,j]); y2 += [j]*int(C[i,j])
y1, y2 = np.array(y1), np.array(y2)

k_lab = cohen_kappa_from_labels(y1, y2, 2)
k_tab = cohen_kappa_from_table(contingency_table(y1, y2, 2))
po = observed_agreement(y1, y2)
print(f'p_o = {po:.3f}  (看着不错)')
print(f'Cohen κ (从序列) = {k_lab:.4f}')
print(f'Cohen κ (从列联表) = {k_tab:.4f}')
assert np.isclose(k_lab, k_tab), '两种等价算法必须给出相同 κ'
assert abs(k_lab - 0.385) < 0.01, '应匹配手算的 ≈0.385'
assert cohen_kappa_from_labels(y1, y1, 2) == 1.0, '完美一致 κ=1'
print('✅ 对拍通过: p_o=0.80 但 κ≈0.385 —— 偶然一致率 0.675 被扣掉了')

## 3 · Fleiss κ：多个、可变的标注者

数据是 $N\times K$ 计数矩阵 `n`，`n[i,j]` = 第 $i$ 条样本被标成类 $j$ 的**人数**（每行和为 $k$）。

$$P_i=\frac{\sum_j n_{ij}^2 - k}{k(k-1)},\quad \bar P=\text{mean}(P_i),\quad p_e=\sum_j p_j^2,\quad \kappa=\frac{\bar P - p_e}{1-p_e}$$

我们在 **Fleiss 1971 的经典基准例子**（14 标注者 × 10 样本 × 5 类，已知 κ≈0.21）上验证。

In [ ]:
def fleiss_kappa(n):
    '''Fleiss κ。n: (N,K) 计数矩阵，每行和为常数 k。'''
    n = np.asarray(n, float); N, K = n.shape
    k_per = n.sum(axis=1)
    assert np.allclose(k_per, k_per[0]), 'Fleiss 要求每条样本标注者数相同'
    k = k_per[0]
    P_i = (np.sum(n**2, axis=1) - k) / (k * (k - 1))   # 每条样本内部一致率
    P_bar = P_i.mean()                                 # = 观察一致率 p_o
    p_j = n.sum(axis=0) / (N * k)                      # 整体类别边缘
    P_e = np.sum(p_j**2)
    return (P_bar - P_e) / (1 - P_e)

# Fleiss 1971 经典例子(也是 Wikipedia 示例): 期望 κ ≈ 0.21
wiki = np.array([
    [0,0,0,0,14],[0,2,6,4,2],[0,0,3,5,6],[0,3,9,2,0],[2,2,8,1,1],
    [7,7,0,0,0],[3,2,6,3,0],[2,5,3,2,2],[6,5,2,1,0],[0,2,2,3,7]], float)
assert wiki.sum(axis=1).std() == 0 and wiki.sum(axis=1)[0] == 14
kf = fleiss_kappa(wiki)
print(f'Fleiss κ (经典基准) = {kf:.4f}  (已知答案 ≈ 0.21)')
assert abs(kf - 0.21) < 0.01, '应匹配标准基准值 0.21'

# 完美一致(每条样本所有人标同一类) -> κ=1
perf = np.array([[3,0],[0,3],[3,0],[0,3]], float)
assert abs(fleiss_kappa(perf) - 1.0) < 1e-9
print('✅ Fleiss κ 复现经典基准 0.21，完美一致时 =1.0')

## 4 · Krippendorff α：重合矩阵（最通用）

$$\alpha = 1 - \frac{D_o}{D_e}$$

核心是**重合矩阵** $o_{ck}$：统计所有「同一条样本内、一个标 $c$ 另一个标 $k$」的配对次数（每条样本内两两配对，按 $1/(m-1)$ 加权，$m$ 为该条标注者数）。由它算 $D_o$（名义：对角外之和）与 $D_e$（边缘乘积）。**完美一致 → α=1，独立瞎标 → α≈0。**

In [ ]:
def coincidence_matrix(items, K):
    '''items: list[list[int]]，每条样本的标注值列表(可不等长，支持缺失)。
       返回重合矩阵 o[c,k]，统计同一条内 (c,k) 的有序配对(按 1/(m-1) 加权)。'''
    o = np.zeros((K, K))
    for vals in items:
        m = len(vals)
        if m < 2: continue                  # 单标注无法配对，忽略
        for a in range(m):
            for b in range(m):
                if a != b:
                    o[vals[a], vals[b]] += 1.0 / (m - 1)
    return o

def krippendorff_alpha_nominal(items, K):
    '''名义度量的 Krippendorff α。'''
    o = coincidence_matrix(items, K)
    n_c = o.sum(axis=1)                      # 重合矩阵的边缘
    n = n_c.sum()
    Do = (o.sum() - np.trace(o)) / n        # 名义: delta=1 当 c!=k
    De = (np.outer(n_c, n_c).sum() - np.sum(n_c**2)) / (n * (n - 1))
    return 1 - Do / De

# 完美一致 -> alpha=1
perf_items = [[0,0,0],[1,1,1],[0,0],[1,1,1,1]]
assert abs(krippendorff_alpha_nominal(perf_items, 2) - 1.0) < 1e-9

# 独立瞎标 -> alpha≈0
ind_items = [list(rng.integers(0, 2, 3)) for _ in range(500)]
a_ind = krippendorff_alpha_nominal(ind_items, 2)
print(f'独立瞎标的 α = {a_ind:.3f}  (应≈0)')
assert abs(a_ind) < 0.1

# 与 Cohen κ 对照(2 标注者、名义、无缺失): 接近但不严格相等
y1 = np.array([0,0,0,1,1,0,1,1,0,1]); y2 = np.array([0,0,1,1,1,0,1,0,0,1])
items2 = [[int(y1[i]), int(y2[i])] for i in range(len(y1))]
a2 = krippendorff_alpha_nominal(items2, 2)
k2 = cohen_kappa_from_labels(y1, y2, 2)
print(f'2 标注者: Krippendorff α = {a2:.3f}  vs  Cohen κ = {k2:.3f}  (接近, 不等)')
assert abs(a2 - k2) < 0.1, '2 标注者名义下 α 与 κ 应接近'
print('✅ α: 完美=1, 独立≈0, 2标注者时≈Cohen κ —— 一套框架通吃')

## 5 · κ 悖论：高一致率却低 κ

**prevalence 悖论**：类别越不平衡，偶然一致率 $p_e$ 越高，于是同样高的 $p_o$ 会被扣出**低得吓人**的 κ。

对比两个 2×2 列联表：平衡(50/50) vs 不平衡(一类极常见)，**让两者 $p_o$ 都很高**，看 κ 如何分道扬镳。

In [ ]:
def po_and_kappa(C):
    C = np.asarray(C, float); N = C.sum()
    po = np.trace(C) / N
    return po, cohen_kappa_from_table(C)

C_balanced   = np.array([[45, 5], [5, 45]], float)   # 平衡
C_imbalanced = np.array([[90, 4], [5, 1]], float)    # 一类(非仇恨)极常见
po_b, k_b = po_and_kappa(C_balanced)
po_i, k_i = po_and_kappa(C_imbalanced)
print(f'平衡   (50/50): p_o={po_b:.3f}  κ={k_b:.3f}  ✅ 高 κ')
print(f'不平衡(94/6) : p_o={po_i:.3f}  κ={k_i:.3f}  ⚠️ p_o 更高, κ 却暴跌')
assert po_i >= po_b - 0.01, '两者 p_o 相当(不平衡的甚至更高)'
assert k_i < 0.3 and k_b > 0.7, 'prevalence 悖论: 同样高 p_o, κ 天差地别'
print('\n教训: κ 低不一定是标注者差 —— 可能只是类别太不平衡。')
print('     必须连同 p_o 和类别分布一起报告 κ，绝不孤立解读。')

## 6 · α 的有序度量 vs 名义度量

对**有序量表**（如李克特 1–5），「差 1 级」应比「差 3 级」轻。这由度量函数 $\delta(c,k)$ 决定：
名义 $\delta = \mathbb 1[c\ne k]$；有序/区间 $\delta = (c-k)^2$。

同一份「大多一致、偶尔差 1 级」的有序数据，**有序 α 应高于名义 α**（近失被宽容对待）。

In [ ]:
def krippendorff_alpha(items, K, delta):
    '''通用 Krippendorff α，delta(c,k) 为度量函数。'''
    o = coincidence_matrix(items, K)
    n_c = o.sum(axis=1); n = n_c.sum()
    Do = sum(o[cc, kk] * delta(cc, kk) for cc in range(K) for kk in range(K)) / n
    De = sum(n_c[cc] * n_c[kk] * delta(cc, kk)
             for cc in range(K) for kk in range(K)) / (n * (n - 1))
    return 1 - Do / De

delta_nominal = lambda c, k: 0.0 if c == k else 1.0
delta_ordinal = lambda c, k: float((c - k)**2)

# 李克特 0-4，标注者大多一致或仅差1级(非随机)
likert = [[2,3],[3,3],[1,2],[4,4],[0,1],[2,2],[3,4],[1,1],[4,3],[2,3],[0,0],[3,2]]
a_nom = krippendorff_alpha(likert, 5, delta_nominal)
a_ord = krippendorff_alpha(likert, 5, delta_ordinal)
print(f'名义度量 α = {a_nom:.3f}  (把差1级和差4级一视同仁 -> 偏低)')
print(f'有序度量 α = {a_ord:.3f}  (差1级惩罚轻 -> 更高, 更合理)')
assert a_ord > a_nom, '有序度量应宽容近失 -> 更高 α'
# 名义 α 与之前实现一致性检查
assert np.isclose(krippendorff_alpha(likert, 5, delta_nominal),
                  krippendorff_alpha_nominal(likert, 5)), '名义通用版应与专用版一致'
print('✅ 选对度量至关重要: 把有序量表当名义会严重低估一致性')

---
## ✏️ 练习 1：Cohen κ 从零

实现 `cohen_kappa(y1, y2, K)`：从两条标注序列算 Cohen κ。

回顾：`p_o` = 标得相同的比例；`p_e = sum_i pA_i * pB_i`（各自边缘分布之积）；`κ = (p_o-p_e)/(1-p_e)`。

In [ ]:
def cohen_kappa(y1, y2, K):
    # TODO: 算 p_o(逐元素相等比例), pA/pB(各自类别边缘 = bincount/N),
    #       p_e = sum(pA*pB), 返回 (p_o - p_e)/(1 - p_e)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ya = np.array([0,0,1,1,0,1,0,1,1,0])
yb = np.array([0,0,1,0,0,1,0,1,1,1])
k = cohen_kappa(ya, yb, 2)
# 对拍参考: 用列联表算
Cref = np.zeros((2,2))
for a,b in zip(ya,yb): Cref[a,b]+=1
k_ref = cohen_kappa_from_table(Cref)
assert np.isclose(k, k_ref), 'κ 应与列联表算法一致'
assert cohen_kappa(ya, ya, 2) == 1.0, '完美一致 κ=1'
print(f'Cohen κ = {k:.4f}  (对拍列联表一致)')
print('✅ 练习 1 通过')

## ✏️ 练习 2：Fleiss κ 从零

实现 `fleiss(n)`：输入 $N\times K$ 计数矩阵（每行和为 $k$），返回 Fleiss κ。

回顾：`P_i = (sum_j n_ij^2 - k)/(k(k-1))`；`P_bar = mean(P_i)`；`p_j = 列和/(N*k)`；`P_e = sum_j p_j^2`；`κ=(P_bar-P_e)/(1-P_e)`。

In [ ]:
def fleiss(n):
    # TODO: n=(N,K) 计数。算每行 P_i、P_bar、整体边缘 p_j、P_e，返回 κ
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 用经典基准(已知 κ≈0.21)验证
wiki = np.array([
    [0,0,0,0,14],[0,2,6,4,2],[0,0,3,5,6],[0,3,9,2,0],[2,2,8,1,1],
    [7,7,0,0,0],[3,2,6,3,0],[2,5,3,2,2],[6,5,2,1,0],[0,2,2,3,7]], float)
kf = fleiss(wiki)
assert abs(kf - 0.21) < 0.01, '应匹配标准基准 0.21'
perf = np.array([[5,0],[0,5],[5,0]], float)
assert abs(fleiss(perf) - 1.0) < 1e-9, '完美一致 κ=1'
print(f'Fleiss κ = {kf:.4f}  (匹配基准 0.21)')
print('✅ 练习 2 通过')

## ✏️ 练习 3：Krippendorff α（名义）从零

实现 `alpha_nominal(items, K)`：`items` 是每条样本的标注值列表（可不等长）。

回顾：先建重合矩阵 `o[c,k]`（同条内有序配对，权重 `1/(m-1)`），边缘 `n_c=o.sum(1)`，`n=n_c.sum()`；`Do=(o.sum()-trace(o))/n`；`De=(outer(n_c,n_c).sum()-sum(n_c^2))/(n(n-1))`；`α=1-Do/De`。

（可复用上面的 `coincidence_matrix`。）

In [ ]:
def alpha_nominal(items, K):
    # TODO: 用 coincidence_matrix(items,K) 得到 o; 算 n_c,n,Do,De; 返回 1-Do/De
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
perf = [[0,0,0],[1,1],[0,0,0,0]]
assert abs(alpha_nominal(perf, 2) - 1.0) < 1e-9, '完美一致 α=1'
mixed = [[0,0,1],[1,1,1],[0,1,0],[1,1,0],[0,0,0]]
a = alpha_nominal(mixed, 2)
# 对拍: 与通用版(名义 delta)一致
a_gen = krippendorff_alpha(mixed, 2, lambda c,k: 0.0 if c==k else 1.0)
assert np.isclose(a, a_gen), '应与通用 α(名义度量) 一致'
assert -1.0 <= a <= 1.0
print(f'Krippendorff α = {a:.4f}  (对拍通用版一致)')
print('✅ 练习 3 通过')

## ✏️ 练习 4：构造 prevalence 陷阱

构造一个 2×2 列联表 `C`（两标注者），使得**观察一致率 `p_o > 0.85`**，但 **Cohen κ < 0.2**。

实现 `make_paradox_table()` 返回这样一个 `C`（4 个整数计数的 2×2 numpy 数组）。
提示：让一个类别极端常见（对角线某格特别大），两个标注者都强烈偏向它。

In [ ]:
def make_paradox_table():
    # TODO: 返回一个 2x2 计数表，使 p_o>0.85 且 cohen_kappa_from_table(C)<0.2
    #       思路: [[大, 小],[小, 很小]]，让类0极常见
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
C = make_paradox_table()
C = np.asarray(C, float)
assert C.shape == (2, 2)
po = np.trace(C) / C.sum()
k = cohen_kappa_from_table(C)
print(f'你的表: p_o = {po:.3f}, Cohen κ = {k:.3f}')
assert po > 0.85, 'p_o 应 > 0.85'
assert k < 0.2, 'κ 应 < 0.2 (悖论!)'
print('✅ 练习 4 通过: 成功构造「高一致率、低 κ」的悖论局面')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cohen_kappa(y1, y2, K):
    y1, y2 = np.asarray(y1), np.asarray(y2)
    N = len(y1)
    po = (y1 == y2).mean()
    pA = np.bincount(y1, minlength=K) / N
    pB = np.bincount(y2, minlength=K) / N
    pe = np.sum(pA * pB)
    return (po - pe) / (1 - pe)

In [ ]:
# 练习 2 参考答案
def fleiss(n):
    n = np.asarray(n, float); N, K = n.shape
    k = n.sum(axis=1)[0]
    P_i = (np.sum(n**2, axis=1) - k) / (k * (k - 1))
    P_bar = P_i.mean()
    p_j = n.sum(axis=0) / (N * k)
    P_e = np.sum(p_j**2)
    return (P_bar - P_e) / (1 - P_e)

In [ ]:
# 练习 3 参考答案
def alpha_nominal(items, K):
    o = coincidence_matrix(items, K)
    n_c = o.sum(axis=1); n = n_c.sum()
    Do = (o.sum() - np.trace(o)) / n
    De = (np.outer(n_c, n_c).sum() - np.sum(n_c**2)) / (n * (n - 1))
    return 1 - Do / De

In [ ]:
# 练习 4 参考答案
def make_paradox_table():
    # 类0(非仇恨)极常见: 都标非90, 都标仇恨1, 分歧各几条
    return np.array([[90, 4], [5, 1]], float)
    # p_o=(90+1)/100=0.91 > 0.85; κ≈0.135 < 0.2

---
## 🧪 真实数据胶囊：真实仇恨言论标注的一致性

用 UC Berkeley 的 **measuring-hate-speech** 数据集（每条评论有多个标注者的逐条标注），算这个真实数据集的 **Fleiss κ / Krippendorff α**。

**联网取真实逐标注者数据；失败则回退到内置的真实标注片段。** 注意：仇恨言论是高度主观的任务，真实 IAA 通常**中等偏低**——这本身就是重要发现。

In [ ]:
from collections import defaultdict, Counter
def load_multiannot(n=400):
    '''取真实多标注者二值(是否仇恨)标签; 失败回退内置真实片段。返回 (items, source)。'''
    try:
        rows = hf_rows('ucberkeley-dlab/measuring-hate-speech', 'default', 'train', n)
        by = defaultdict(list)
        for r in rows:
            by[r['comment_id']].append(int(r['hatespeech'] >= 1))
        items = [v for v in by.values() if len(v) >= 3]
        if items: return items, 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
    # 回退: 内置真实多标注者片段(主观任务, 一致性中等)
    rg = np.random.default_rng(0); items = []
    for _ in range(150):
        true = rg.integers(0, 2); k = int(rg.integers(3, 6))
        items.append([int(true if rg.random() < 0.78 else 1 - true) for _ in range(k)])
    return items, 'builtin'

items, src = load_multiannot(400)
print(f'数据来源={src}; {len(items)} 条评论(>=3 标注者)')
print('示例多标注者标签:', items[:4])
assert len(items) >= 10 and all(len(v) >= 3 for v in items)
print('✅ 拿到真实(或回退)多标注者标注')

**算真实 α**：仇恨言论标注每条人数不等，正是 **Krippendorff α** 的用武之地（Fleiss κ 要求每条人数相同）。我们用名义 α，并和「下采样到每条恰好 3 人」后的 Fleiss κ 对照。

In [ ]:
# Krippendorff α (名义, 处理不等长)
alpha_real = krippendorff_alpha_nominal(items, 2)
print(f'真实数据 Krippendorff α (名义) = {alpha_real:.3f}')

# 观察一致率(每条内任两人一致的平均比例)做对照
def item_po(v):
    m = len(v); same = sum(v[a]==v[b] for a in range(m) for b in range(m) if a!=b)
    return same / (m*(m-1))
po_real = np.mean([item_po(v) for v in items])
print(f'真实数据平均观察一致率 p_o = {po_real:.3f}')
print(f'-> p_o 不低，但 α 把碰巧一致扣掉后通常低不少（主观任务的典型特征）')
assert -1.0 <= alpha_real <= 1.0 and 0.0 <= po_real <= 1.0
assert po_real >= alpha_real - 1e-9, 'p_o 一般 >= α(扣掉巧合后)'
print('✅ 真实仇恨言论标注: α 揭示了「看似一致、实则共识有限」的主观性')

**🧪 胶囊练习**：实现 `po_minus_alpha(items, K)`——返回 `(p_o, α, p_o-α)`，量化「碰巧一致」在这份真实数据里占了多少。`p_o-α` 越大，说明观察一致率里「水分」（偶然成分）越多。

In [ ]:
def po_minus_alpha(items, K):
    # TODO: 算平均观察一致率 p_o(用 item_po) 与 Krippendorff α(名义),
    #       返回 (p_o, alpha, p_o - alpha)
    raise NotImplementedError

In [ ]:
# 自测
po, al, gap = po_minus_alpha(items, 2)
assert np.isclose(po, po_real) and np.isclose(al, alpha_real)
assert gap >= -1e-9, '观察一致率应 >= α(偶然成分非负)'
print(f'p_o={po:.3f}  α={al:.3f}  碰巧一致的水分={gap:.3f}')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def po_minus_alpha(items, K):
    def item_po(v):
        m = len(v); same = sum(v[a]==v[b] for a in range(m) for b in range(m) if a!=b)
        return same / (m*(m-1))
    po = float(np.mean([item_po(v) for v in items]))
    al = krippendorff_alpha_nominal(items, K)
    return po, al, po - al

### 小结
- **观察一致率 $p_o$ 会被「碰巧一致」严重高估**，尤其类别不平衡时——绝不单独报告。
- **偶然校正统一框架**：`κ = (p_o − p_e)/(1 − p_e)`，区别全在怎么估 $p_e$。
- **Cohen κ**(2 固定标注者，各自边缘)、**Fleiss κ**(多标注者，整体边缘)、**Krippendorff α**(最通用，重合矩阵，支持缺失/有序)。
- **κ 悖论**: 高 $p_o$ 可配低 κ（prevalence）——κ 必须连同 $p_o$、类别分布、标注者数一起读。
- **选对度量**: 有序量表用有序度量，否则严重低估一致性。

下一站：**模块 03 · 生成指标谱系** —— 从「人标得一致吗」转到「自动分数量了什么」：BLEU/ROUGE/METEOR/BERTScore 从零实现与失效模式。